In [1]:
import pandas as pd

# load all stair-relevant attribute train files
train_files = {
    "fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "material": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

# filter each to stairs only
stair_assets = {}
for attr, df in train_files.items():
    stair_assets[attr] = set(
        df[df["profile_name"] == "Stairs"]["asset_id"].unique()
    )

# find overlap across all 5
overlap = set.intersection(*stair_assets.values())
print(f"Assets in all 5 stair attribute train files: {len(overlap)}")
print(list(overlap)[:5])

Assets in all 5 stair attribute train files: 3
[np.int64(126777), np.int64(121499), np.int64(121503)]


In [2]:
test_asset_id = 126777

# verify it exists in all files
for attr, df in train_files.items():
    row = df[df["asset_id"] == test_asset_id]
    print(f"{attr}: {row[f'attr_{attr}' if attr != 'material' else 'attr_material_frame,_tank,_body'].values}")

fall_height: [0.5]
has_pedestrian_railing: ['1 railing']
material: ['Timber/Wood']
number_of_steps: [12.]
structure_position: ['Elevated']


In [3]:
# use structure position train file since it has all columns including image_path
test_df = train_files["structure_position"][
    train_files["structure_position"]["asset_id"] == test_asset_id
]

print(f"Images for asset {test_asset_id}: {len(test_df)}")
test_df[["asset_id", "profile_name", "image_path"]].to_csv(
    "../data/processed/train/pipeline_test_asset.csv", index=False
)

Images for asset 126777: 1


In [ ]:
#comparing test with ground truth labels

train_files = {
    "attr_fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "attr_has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "attr_material_frame,_tank,_body": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "attr_number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "attr_structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

for attr, df in train_files.items():
    row = df[df["asset_id"] == 126777]
    if len(row) > 0:
        print(f"{attr}: {row[attr].values[0]}")

attr_fall_height: 0.5
attr_has_pedestrian_railing: 1 railing
attr_material_frame,_tank,_body: Timber/Wood
attr_number_of_steps: 12.0
attr_structure_position: Elevated


In [10]:
fall_height_bin_train = pd.read_csv("../data/processed/train/fall_height_bin_train.csv")
steps_bin_train = pd.read_csv("../data/processed/train/steps_bin_train.csv")

print("fall_height_bin - asset 126777 in train:", 
      126777 in fall_height_bin_train["asset_id"].values)

print("steps_bin - asset 126777 in train:", 
      126777 in steps_bin_train["asset_id"].values)

fall_height_bin - asset 126777 in train: True
steps_bin - asset 126777 in train: True


In [11]:
print(fall_height_bin_train[fall_height_bin_train["asset_id"] == 126777][["asset_id", "fall_height_bin"]])
print(steps_bin_train[steps_bin_train["asset_id"] == 126777][["asset_id", "steps_bin"]])

    asset_id    fall_height_bin
67    126777  medium (0.5-1.2m)
    asset_id       steps_bin
37    126777  medium (10-20)


In [13]:
import pandas as pd
preds = pd.read_csv("../results/test_pipeline_stairs.csv")
print(preds.columns.tolist())
print(preds.iloc[0])

['asset_id', 'timestamp', 'model', 'response', 'fall_height_bin_value', 'fall_height_bin_confidence', 'has_pedestrian_railing_value', 'has_pedestrian_railing_confidence', 'material_frame_tank_body_value', 'material_frame_tank_body_confidence', 'steps_bin_value', 'steps_bin_confidence', 'structure_position_value', 'structure_position_confidence']
asset_id                                                                          126777
timestamp                                                     2026-05-19T21:27:46.851836
model                                                             gemini-3-flash-preview
response                               {\n    "fall_height": {\n    "value": "high (>...
fall_height_bin_value                                                       high (>1.2m)
fall_height_bin_confidence                                                           0.9
has_pedestrian_railing_value                                                   1 railing
has_pedestrian_railing_confid

In [6]:
PROMPT_TEMPLATE = """
    You are an expert in park infrastructure analysis.

    Using ALL provided images of this single stair asset, identify the most likely
    attribute values. For each of the following attributes, the possible values are
    given below. Predict exactly ONE value from the listed options for each
    attribute, and provide a confidence score (0.0-1.0) for each prediction.

    Attributes to predict:
    - fall_height: low (<0.5m) | medium (0.5m-1.2m) | high (>1.2m)
    - has_pedestrian_railing: 2 railings | 1 railing | no railings
    - material_frame_tank_body: PVC | Gravel | Natural Surface | Earth-filled |
                                Aluminum | Metal | Steel | Rock/Stone | Concrete |
                                Box Step | Timber/Wood
    - number_of_steps: few (<10) | medium (10-20) | many (>20)
    - structure_position: Elevated | At-Grade | Other

    Return ONLY a valid JSON object with this exact schema (no markdown, no prose):
    {
        "<attribute_key>": {
        "value": "<predicted value or 'unable to determine'>",
        "confidence": <float 0.0-1.0>
        }
    }

    If you cannot determine an attribute from the images, set value to
    "unable to determine" and confidence to 0.0.
    """

In [9]:
import sys
from pathlib import Path

# add project root to path
sys.path.insert(0, str(Path("..").resolve()))

from src.vlm.predictors import predict_asset_attributes
import pandas as pd

df = pd.read_csv("../data/processed/train/pipeline_test_asset.csv")

result = predict_asset_attributes(
    asset_id=126777,
    df=df,
    model_name="gemini-3-flash-preview",
    prompt=PROMPT_TEMPLATE
)

print(result)

{'asset_id': 126777, 'error': "503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}", 'response': None}


In [14]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# how many train_only assets have a length label
overlap = train_only[train_only["asset_id"].isin(length_train["asset_id"])]
print(f"Assets with length label in train_only: {len(overlap['asset_id'].unique())}")

Assets with length label in train_only: 819


In [17]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# filter train_only to only assets with length label
length_assets = train_only[train_only["asset_id"].isin(length_train["asset_id"])]
length_assets.to_csv("../data/processed/train/train_only_length_assets.csv", index=False)

print(f"Assets with length label: {length_assets['asset_id'].nunique()}")

Assets with length label: 819


In [18]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

print(f"Total rows: {len(train_only)}")
print(f"Unique asset IDs: {train_only['asset_id'].nunique()}")
print(f"Rows per asset:")
print(train_only.groupby('asset_id').size().describe())

# show assets with more than 1 row
duplicates = train_only[train_only.duplicated('asset_id', keep=False)]
print(f"\nAssets with multiple rows: {duplicates['asset_id'].nunique()}")
print(duplicates[['asset_id', 'profile_name', 'image_path']].head(10))

Total rows: 2582
Unique asset IDs: 1687
Rows per asset:
count    1687.000000
mean        1.530528
std         1.094085
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        10.000000
dtype: float64

Assets with multiple rows: 517
    asset_id           profile_name  \
0      47664  Boardwalk < 1.2m High   
2      47664  Boardwalk < 1.2m High   
3      48119  Boardwalk < 1.2m High   
4      47664  Boardwalk < 1.2m High   
5      48119  Boardwalk < 1.2m High   
6      48119  Boardwalk < 1.2m High   
9      48934  Boardwalk < 1.2m High   
11     48934  Boardwalk < 1.2m High   
14     51547  Boardwalk < 1.2m High   
15     51547  Boardwalk < 1.2m High   

                                           image_path  
0   data/citywide/images/337/47664/86079__AST_EX_2...  
2   data/citywide/images/337/47664/86081__AST_EX_2...  
3   data/citywide/images/337/48119/13815__MI_20210...  
4   data/citywide/images/337/47664/86084__AST_EX_2...  
5   data/citywide/i

In [19]:
# check if attribute columns are identical across duplicate rows
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# get assets with multiple rows
multi_image_assets = train_only[train_only.duplicated('asset_id', keep=False)]['asset_id'].unique()

# check if labels are consistent across rows for same asset
inconsistent = []
for asset_id in multi_image_assets:
    rows = length_train[length_train['asset_id'] == asset_id]
    if len(rows) > 1:
        # check if length_bin is the same across all rows
        if rows['length_bin'].nunique() > 1:
            inconsistent.append(asset_id)

print(f"Assets with inconsistent length_bin labels: {len(inconsistent)}")
if len(inconsistent) > 0:
    print(inconsistent)
else:
    print("✅ All duplicate assets have consistent labels across rows")

Assets with inconsistent length_bin labels: 0
✅ All duplicate assets have consistent labels across rows


In [20]:
import pandas as pd
import glob

batches = glob.glob("../results/vlm_length_*.csv")
combined = pd.concat([pd.read_csv(f) for f in sorted(batches)])
combined = combined.drop_duplicates("asset_id")
combined.to_csv("../results/vlm_length_combined.csv", index=False)
print(f"Total assets: {len(combined)}")

Total assets: 70


In [21]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

stairs = train_only[train_only["profile_name"] == "Stairs"]
print(f"Total stair assets in train_only: {stairs['asset_id'].nunique()}")

Total stair assets in train_only: 446


In [22]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
sp_train = pd.read_csv("../data/processed/train/attr_structure_position_train.csv")

# filter to stairs assets that are in train_only AND have structure position label
stairs_sp = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["asset_id"].isin(sp_train["asset_id"]))
]

print(f"Stair assets with structure position label in train_only: {stairs_sp['asset_id'].nunique()}")

# save
#stairs_sp.to_csv("../data/processed/train/train_only_stairs_sp_assets.csv", index=False)

Stair assets with structure position label in train_only: 402


In [24]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

stairs_sp = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["attr_structure_position"].notna())
]

print(f"Stair assets with structure position label: {stairs_sp['asset_id'].nunique()}")
stairs_sp.to_csv("../data/processed/train/train_only_stairs_sp_assets.csv", index=False)

Stair assets with structure position label: 402


In [25]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

stairs_assets_with_faces = [51544, 89719, 121503, 121829]

stairs_sp = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["attr_structure_position"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]

print(f"Stair assets with structure position label (no faces): {stairs_sp['asset_id'].nunique()}")
stairs_sp.to_csv("../data/processed/train/train_only_stairs_sp_assets.csv", index=False)

Stair assets with structure position label (no faces): 398


In [26]:
from pathlib import Path

image_dir = Path("../data/raw/citywide/images")
images = list(image_dir.rglob("*.jpeg")) + \
         list(image_dir.rglob("*.jpg")) + \
         list(image_dir.rglob("*.png")) + \
         list(image_dir.rglob("*.JPG")) + \
         list(image_dir.rglob("*.JPEG"))

print(f"Total images on disk: {len(images)}")

Total images on disk: 5310


In [27]:
import pandas as pd

master = pd.read_csv("../data/processed/master_dataset.csv")

print(f"Total rows: {len(master)}")
print(f"Unique asset IDs: {master['asset_id'].nunique()}")
print(f"Duplicate rows: {master.duplicated().sum()}")
print(f"Duplicate asset+image combos: {master.duplicated(['asset_id', 'filename']).sum()}")

Total rows: 5562
Unique asset IDs: 3584
Duplicate rows: 0
Duplicate asset+image combos: 37


In [30]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
steps_bin_train = pd.read_csv("../data/processed/train/steps_bin_train.csv")

stairs_assets_with_faces = [51544, 89719, 121503, 121829]

# filter stairs with steps_bin label and no faces
stairs_steps = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["steps_bin"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]

print(f"Stair assets with steps_bin label (no faces): {stairs_steps['asset_id'].nunique()}")

# verify against steps_bin train csv
steps_bin_assets = set(steps_bin_train[steps_bin_train["profile_name"] == "Stairs"]["asset_id"].unique())
train_only_assets = set(stairs_steps["asset_id"].unique())

print(f"Assets in steps_bin_train (stairs): {len(steps_bin_assets)}")
print(f"Overlap: {len(steps_bin_assets & train_only_assets)}")

stairs_steps.to_csv("../data/processed/train/train_only_stairs_steps_assets.csv", index=False)

Stair assets with steps_bin label (no faces): 19
Assets in steps_bin_train (stairs): 29
Overlap: 19


In [29]:
stairs_steps_simple = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["steps_bin"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]

print(f"Stair assets with steps_bin label (no faces): {stairs_steps_simple['asset_id'].nunique()}")

Stair assets with steps_bin label (no faces): 19


In [34]:
stairs_assets_with_faces = [51544, 89719, 121503, 121829]

stairs_fall_height = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["fall_height_bin"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]

print(f"Stair assets with fall_height_bin label (no faces): {stairs_fall_height['asset_id'].nunique()}")
stairs_fall_height.to_csv("../data/processed/train/train_only_stairs_fall_height_assets.csv", index=False)

Stair assets with fall_height_bin label (no faces): 5


In [45]:
stairs_assets_with_faces = [51544, 89719, 121503, 121829]

# material
stairs_material = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["attr_material_frame,_tank,_body"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]
print(f"Stair assets with material label (no faces): {stairs_material['asset_id'].nunique()}")
stairs_material.to_csv("../data/processed/train/train_only_stairs_material_assets.csv", index=False)

# has pedestrian railing
stairs_railing = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["attr_has_pedestrian_railing"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]
print(f"Stair assets with railing label (no faces): {stairs_railing['asset_id'].nunique()}")
stairs_railing.to_csv("../data/processed/train/train_only_stairs_railing_assets.csv", index=False)

Stair assets with material label (no faces): 423
Stair assets with railing label (no faces): 376


In [35]:
import pandas as pd

stairs_assets_with_faces = [51544, 89719, 121503, 121829]

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

# filter to stairs with ALL stair attributes labelled and no faces
stairs_all_attrs = train_only[
    (train_only["profile_name"] == "Stairs") &
    (train_only["attr_structure_position"].notna()) &
    (train_only["attr_has_pedestrian_railing"].notna()) &
    (train_only["attr_material_frame,_tank,_body"].notna()) &
    (train_only["fall_height_bin"].notna()) &
    (train_only["steps_bin"].notna()) &
    (~train_only["asset_id"].isin(stairs_assets_with_faces))
]

print(f"Stair assets with ALL attributes labelled: {stairs_all_attrs['asset_id'].nunique()}")

Stair assets with ALL attributes labelled: 2


In [37]:
stairs_sp = pd.read_csv("../data/processed/train/train_only_stairs_sp_assets.csv")
sample_20 = stairs_sp.drop_duplicates("asset_id").head(20)
shared_asset_ids = sample_20["asset_id"].tolist()
print(shared_asset_ids)

# save shared sample
shared_sample = stairs_sp[stairs_sp["asset_id"].isin(shared_asset_ids)]
shared_sample.to_csv("../data/processed/train/train_only_stairs_gemini_sample.csv", index=False)

[47049, 47050, 47053, 47054, 47184, 48269, 48347, 48394, 48395, 48799, 48937, 49501, 49525, 49702, 49703, 49704, 49705, 49707, 49710, 49711]


In [38]:
stairs_sp = pd.read_csv("../data/processed/train/train_only_stairs_sp_assets.csv")
sample_20 = stairs_sp.drop_duplicates("asset_id").head(20)

print(f"Total assets: {sample_20['asset_id'].nunique()}")
print(f"\nLabel coverage for these 20 assets:")
print(f"attr_structure_position: {sample_20['attr_structure_position'].notna().sum()}")
print(f"attr_has_pedestrian_railing: {sample_20['attr_has_pedestrian_railing'].notna().sum()}")
print(f"attr_material_frame,_tank,_body: {sample_20['attr_material_frame,_tank,_body'].notna().sum()}")
print(f"fall_height_bin: {sample_20['fall_height_bin'].notna().sum()}")
print(f"steps_bin: {sample_20['steps_bin'].notna().sum()}")

Total assets: 20

Label coverage for these 20 assets:
attr_structure_position: 20
attr_has_pedestrian_railing: 19
attr_material_frame,_tank,_body: 19
fall_height_bin: 0
steps_bin: 0


In [39]:
import pandas as pd

preds = pd.read_csv("../results/vlm_structure_position_gemini_preview_sample.csv")

print(f"Total rows: {len(preds)}")
print(f"Parse errors: {preds['parse_error'].sum()}")
print(f"Successful predictions: {preds['structure_position_value'].notna().sum()}")

# check which assets had errors
error_assets = preds[preds["parse_error"] == True]["asset_id"].tolist()
print(f"\nAssets with parse errors: {error_assets}")

Total rows: 20
Parse errors: 17
Successful predictions: 3

Assets with parse errors: [47049, 47053, 47184, 48269, 48347, 48394, 48395, 48799, 48937, 49501, 49525, 49702, 49703, 49705, 49707, 49710, 49711]


In [40]:
train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

for asset_id in error_assets[:5]:
    rows = train_only[train_only["asset_id"] == asset_id]
    print(f"\nAsset {asset_id}:")
    print(rows[["asset_id", "image_path", "file_exists"]].to_string(index=False))


Asset 47049:
 asset_id                                                        image_path  file_exists
    47049 data/citywide/images/356/47049/43882__AST_EX_20220601_110955.jpeg         True
    47049 data/citywide/images/356/47049/75505__AST_EX_20241022_111626.jpeg         True

Asset 47053:
 asset_id                                                        image_path  file_exists
    47053      data/citywide/images/356/47053/43873__MI_20220601_102208.jpg         True
    47053 data/citywide/images/356/47053/71435__AST_EX_20240725_113728.jpeg         True

Asset 47184:
 asset_id                                         image_path  file_exists
    47184 data/citywide/images/356/47184/34371__IMG_8323.JPG         True

Asset 48269:
 asset_id                                                        image_path  file_exists
    48269 data/citywide/images/356/48269/21402__AST_EX_20210911_135039.jpeg         True

Asset 48347:
 asset_id                                                        image

In [43]:
import os
from pathlib import Path

ROOT = Path("..").resolve()

sample = pd.read_csv("../data/processed/train/train_only_stairs_gemini_sample.csv")

for asset_id in sample["asset_id"].unique():
    asset_rows = sample[sample["asset_id"] == asset_id]
    images_found = []
    for path in asset_rows["image_path"].tolist():
        fixed = ROOT / path.replace("data/", "data/raw/", 1)
        images_found.append(fixed.exists())
    
    if not any(images_found):
        print(f"⚠️ Asset {asset_id}: NO images found locally")
    else:
        print(f"✅ Asset {asset_id}: {sum(images_found)}/{len(images_found)} images found")

✅ Asset 47049: 2/2 images found
✅ Asset 47050: 1/1 images found
✅ Asset 47053: 2/2 images found
✅ Asset 47054: 2/2 images found
✅ Asset 47184: 1/1 images found
✅ Asset 48269: 1/1 images found
✅ Asset 48347: 1/1 images found
✅ Asset 48394: 1/1 images found
✅ Asset 48395: 1/1 images found
✅ Asset 48799: 1/1 images found
✅ Asset 48937: 1/1 images found
✅ Asset 49501: 1/1 images found
✅ Asset 49525: 1/1 images found
✅ Asset 49702: 4/4 images found
✅ Asset 49703: 2/2 images found
✅ Asset 49704: 1/1 images found
✅ Asset 49705: 1/1 images found
✅ Asset 49707: 3/3 images found
✅ Asset 49710: 2/2 images found
✅ Asset 49711: 7/7 images found


In [44]:
import pandas as pd

preds = pd.read_csv("../results/vlm_pedestrian_railing_gemini_preview_sample.csv")
gt = pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv")
sample = pd.read_csv("../data/processed/train/train_only_stairs_gemini_sample.csv")

# which asset has no label
no_label = sample[~sample["asset_id"].isin(gt["asset_id"])]["asset_id"].tolist()
print(f"Asset with no railing label: {no_label}")

# which asset has parse error
parse_errors = preds[preds["parse_error"] == True]["asset_id"].tolist() if "parse_error" in preds.columns else []
print(f"Assets with parse errors: {parse_errors}")

# are they the same?
print(f"Overlap: {set(no_label) & set(parse_errors)}")

Asset with no railing label: [48395]
Assets with parse errors: [48395]
Overlap: {48395}


In [47]:
import pandas as pd

stairs_sp = pd.read_csv("../data/processed/train/train_only_stairs_sp_assets.csv")

# get all unique assets excluding the first 20 already processed
already_processed = pd.read_csv("../data/processed/train/train_only_stairs_gemini_sample.csv")["asset_id"].unique()

remaining = stairs_sp[
    ~stairs_sp["asset_id"].isin(already_processed)
].drop_duplicates("asset_id")

print(f"Remaining assets: {remaining['asset_id'].nunique()}")

# sample next 20
next_20 = remaining.head(20)
next_batch = stairs_sp[stairs_sp["asset_id"].isin(next_20["asset_id"])]
next_batch.to_csv("../data/processed/train/train_only_stairs_gemini_sample_batch2.csv", index=False)
print(next_20["asset_id"].tolist())

Remaining assets: 378
[50514, 50723, 50805, 51270, 51458, 51536, 51537, 51695, 51804, 52162, 52248, 52622, 52952, 53110, 53111, 53212, 53214, 53221, 53222, 53428]


In [48]:
next_batch = pd.read_csv("../data/processed/train/train_only_stairs_gemini_sample_batch2.csv")
sample_20 = next_batch.drop_duplicates("asset_id")

print(f"Total assets: {sample_20['asset_id'].nunique()}")
print(f"\nLabel coverage for these 20 assets:")
print(f"attr_structure_position: {sample_20['attr_structure_position'].notna().sum()}")
print(f"attr_has_pedestrian_railing: {sample_20['attr_has_pedestrian_railing'].notna().sum()}")
print(f"attr_material_frame,_tank,_body: {sample_20['attr_material_frame,_tank,_body'].notna().sum()}")
print(f"fall_height_bin: {sample_20['fall_height_bin'].notna().sum()}")
print(f"steps_bin: {sample_20['steps_bin'].notna().sum()}")

Total assets: 20

Label coverage for these 20 assets:
attr_structure_position: 20
attr_has_pedestrian_railing: 10
attr_material_frame,_tank,_body: 19
fall_height_bin: 1
steps_bin: 1


## Combining csvs: Gemini-flash-preview 20-batch sampling predicitons 

In [49]:
import pandas as pd

# combine both batches
batch1 = pd.read_csv("../results/vlm_structure_position_gemini_preview_sample.csv")
batch2 = pd.read_csv("../results/vlm_structure_position_gemini_preview_batch2.csv")

combined = pd.concat([batch1, batch2]).drop_duplicates("asset_id")
combined.to_csv("../results/vlm_structure_position_gemini_preview_combined.csv", index=False)

print(f"Batch 1: {len(batch1)} rows, {batch1['asset_id'].nunique()} assets")
print(f"Batch 2: {len(batch2)} rows, {batch2['asset_id'].nunique()} assets")
print(f"Combined: {len(combined)} rows, {combined['asset_id'].nunique()} assets")

Batch 1: 20 rows, 20 assets
Batch 2: 20 rows, 20 assets
Combined: 40 rows, 40 assets


In [50]:
# combine both batches for railing
batch1_railing = pd.read_csv("../results/vlm_pedestrian_railing_gemini_preview_sample.csv")
batch2_railing = pd.read_csv("../results/vlm_pedestrian_railing_gemini_preview_batch2.csv")

combined_railing = pd.concat([batch1_railing, batch2_railing]).drop_duplicates("asset_id")
combined_railing.to_csv("../results/vlm_pedestrian_railing_gemini_preview_combined.csv", index=False)

print(f"Batch 1: {batch1_railing['asset_id'].nunique()} assets")
print(f"Batch 2: {batch2_railing['asset_id'].nunique()} assets")
print(f"Combined: {combined_railing['asset_id'].nunique()} assets")

Batch 1: 20 assets
Batch 2: 20 assets
Combined: 40 assets


In [51]:
# combine both batches for material
batch1_material = pd.read_csv("../results/vlm_material_gemini_preview_sample.csv")
batch2_material = pd.read_csv("../results/vlm_material_gemini_preview_batch2.csv")

combined_material = pd.concat([batch1_material, batch2_material]).drop_duplicates("asset_id")
combined_material.to_csv("../results/vlm_material_gemini_preview_combined.csv", index=False)

print(f"Batch 1: {batch1_material['asset_id'].nunique()} assets")
print(f"Batch 2: {batch2_material['asset_id'].nunique()} assets")
print(f"Combined: {combined_material['asset_id'].nunique()} assets")

Batch 1: 20 assets
Batch 2: 20 assets
Combined: 40 assets


## Extracting trail birdge attributes

In [53]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

trail_bridge_assets_with_faces = []  # add any if identified

trail_bridge = train_only[
    (train_only["profile_name"] == "Trail Bridge") &
    (~train_only["asset_id"].isin(trail_bridge_assets_with_faces))
]

print(f"Total Trail Bridge assets: {trail_bridge['asset_id'].nunique()}")

# generate per-attribute files
attributes = {
    "abutment_material": "attr_abutment_material",
    "bridge_type": "attr_bridge_type",
    "decking_material": "attr_decking_material",
    "fall_height": "fall_height_bin",
    "has_pedestrian_railing": "attr_has_pedestrian_railing",
    "length": "length_bin",
    "structure_material": "attr_structure_material",
    "width": "width_bin",
}

for name, col in attributes.items():
    filtered = trail_bridge[trail_bridge[col].notna()]
    print(f"{name}: {filtered['asset_id'].nunique()} assets")
    filtered.to_csv(f"../data/processed/train/train_only_bridge_{name}_assets.csv", index=False)

Total Trail Bridge assets: 425
abutment_material: 189 assets
bridge_type: 216 assets
decking_material: 267 assets
fall_height: 27 assets
has_pedestrian_railing: 303 assets
length: 208 assets
structure_material: 230 assets
width: 274 assets


In [54]:
# double checking attribute values

import pandas as pd

# check exact unique values for each attribute in train files
attrs = {
    "attr_has_pedestrian_railing": "attr_has_pedestrian_railing_train.csv",
    "attr_decking_material": "attr_decking_material_train.csv",
    "attr_structure_material": "attr_structure_material_train.csv",
    "attr_abutment_material": "attr_abutment_material_train.csv",
    "attr_bridge_type": "attr_bridge_type_train.csv",
    "attr_has_edge_guard": "attr_has_edge_guard_train.csv",
    "fall_height_bin": "fall_height_bin_train.csv",
    "length_bin": "length_bin_train.csv",
    "width_bin": "width_bin_train.csv",
    "steps_bin": "steps_bin_train.csv",
    "attr_material_frame,_tank,_body": "attr_material_frame_tank_body_train.csv",
    "attr_structure_position": "attr_structure_position_train.csv",
}

for attr, filename in attrs.items():
    df = pd.read_csv(f"../data/processed/train/{filename}")
    print(f"\n{attr}:")
    print(sorted(df[attr].dropna().unique().tolist()))


attr_has_pedestrian_railing:
['1 railing', '2 railings', 'No railings']

attr_decking_material:
['Aluminum', 'Asphalt', 'Composite', 'Concrete', 'Steel', 'Timber']

attr_structure_material:
['Aluminum', 'Concrete', 'Steel', 'Stone', 'Timber']

attr_abutment_material:
['Aluminum Sill Fill', 'Composite', 'Concrete', 'Gabions', 'Steel', 'Timber']

attr_bridge_type:
['Beam', 'Fallen Tree', 'Other', 'Suspension', 'Truss']

attr_has_edge_guard:
['No', 'Yes']

fall_height_bin:
['high (>1.2m)', 'high (>15m)', 'high (>5m)', 'low (<0.5m)', 'low (<1.2m)', 'medium (0.5-1.2m)', 'medium (1.2-15m)', 'medium (1.2-5m)']

length_bin:
['large (>20m)', 'long (>100m)', 'long (>20m)', 'long (>30m)', 'medium (10-20m)', 'medium (10-30m)', 'medium (20-100m)', 'medium (5-20m)', 'medium (6-20m)', 'short (<10m)', 'short (<20m)', 'short (<5m)', 'short (<6m)', 'small (<10m)']

width_bin:
['medium (3-7m)', 'narrow (<0.8m)', 'narrow (<0.9m)', 'narrow (<3m)', 'standard (0.9-1.5m)', 'standard (>=0.8m)', 'wide (>1.5m)'

In [55]:
import pandas as pd

teammate_train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")  # or her specific file

# filter trail bridges with fall height label and file exists
trail_bridge_fall_height = teammate_train_only[
    (teammate_train_only["profile_name"] == "Trail Bridge") &
    (teammate_train_only["fall_height_bin"].notna()) &
    (teammate_train_only["file_exists"] == True)
]

print(f"Trail Bridge assets with fall height label and existing images: {trail_bridge_fall_height['asset_id'].nunique()}")

Trail Bridge assets with fall height label and existing images: 27


In [56]:
import pandas as pd

your_preds = pd.read_csv("../results/vlm_bridge_fall_height_gemini_preview.csv")
gt = pd.read_csv("../data/processed/train/fall_height_bin_train.csv")

print(f"Your predictions: {your_preds['asset_id'].nunique()} assets")
print(f"Parse errors: {your_preds['parse_error'].sum() if 'parse_error' in your_preds.columns else 'no parse_error col'}")
print(f"Successful predictions: {your_preds['fall_height_bin_value'].notna().sum() if 'fall_height_bin_value' in your_preds.columns else 'col missing'}")

# merge
merged = your_preds.merge(gt[["asset_id", "fall_height_bin"]], on="asset_id", how="inner")
merged = merged.drop_duplicates("asset_id")
merged = merged[merged["fall_height_bin_value"].notna()]
merged = merged[merged["fall_height_bin"].notna()]

print(f"After merge and filtering: {len(merged)} assets")

Your predictions: 27 assets
Parse errors: 0.0
Successful predictions: 27
After merge and filtering: 27 assets


In [57]:
teammate_preds = pd.read_csv("../results/vlm_trail_bridge_gemini-3-flash_complete.csv")
gt = pd.read_csv("../data/processed/train/fall_height_bin_train.csv")

print(f"Teammate predictions: {teammate_preds['asset_id'].nunique()} assets")
print(f"Teammate asset types: {teammate_preds['profile_name'].value_counts() if 'profile_name' in teammate_preds.columns else 'no profile_name col'}")

# merge
merged = teammate_preds.merge(gt[["asset_id", "fall_height_bin"]], on="asset_id", how="inner")
merged = merged.drop_duplicates("asset_id")

# check fall height column name in her predictions
print(f"\nTeammate columns: {teammate_preds.columns.tolist()}")

merged = merged[merged["fall_height_bin"].notna()]
print(f"After merge: {len(merged)} assets")
print(f"Assets in yours not hers: {set(your_preds['asset_id']) - set(teammate_preds['asset_id'])}")
print(f"Assets in hers not yours: {set(teammate_preds['asset_id']) - set(your_preds['asset_id'])}")

Teammate predictions: 408 assets
Teammate asset types: no profile_name col

Teammate columns: ['asset_id', 'timestamp', 'model', 'response', 'latency_s', 'abutment_material_value', 'abutment_material_confidence', 'bridge_type_value', 'bridge_type_confidence', 'decking_material_value', 'decking_material_confidence', 'fall_height_bin_value', 'fall_height_bin_confidence', 'has_pedestrian_railing_value', 'has_pedestrian_railing_confidence', 'length_bin_value', 'length_bin_confidence', 'width_bin_value', 'width_bin_confidence', 'structure_position_value', 'structure_position_confidence', 'parse_error', 'raw_response', 'error', 'traceback']
After merge: 24 assets
Assets in yours not hers: {127136, 127147, 123352}
Assets in hers not yours: {92160, 124929, 119811, 92164, 118797, 51214, 51216, 51218, 52242, 88095, 98336, 96291, 96292, 102436, 119851, 51244, 54321, 109619, 54324, 50231, 50235, 88129, 89154, 88131, 88132, 89155, 88135, 88136, 65609, 102471, 113731, 113733, 89163, 47182, 89177, 10

In [58]:
import pandas as pd

file1 = pd.read_csv("../results/vlm_bridge_width_gemma.csv")
file2 = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv")

print(f"File 1 (results/): {len(file1)} rows, {file1['width_bin_value'].notna().sum()} successful")
print(f"File 2 (bridge_attributes/): {len(file2)} rows, {file2['width_bin_value'].notna().sum()} successful")

# check overlap
overlap = set(file1['asset_id']) & set(file2['asset_id'])
print(f"Overlapping assets: {len(overlap)}")

KeyError: 'width_bin_value'

In [59]:
print("File 1 columns:", file1.columns.tolist())
print("File 2 columns:", file2.columns.tolist())

File 1 columns: ['117197', '2026-05-27T22:23:26.726465', 'gemma-4-26b-a4b-it', '{\n    "width_bin": {"value": "standard (0.9-1.5m)", "confidence": 0.85}\n}', '14.459', 'standard (0.9-1.5m)', '0.85', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10']
File 2 columns: ['asset_id', 'timestamp', 'model', 'response', 'latency_s', 'width_bin_value', 'width_bin_confidence', 'parse_error', 'raw_response', 'error', 'traceback']


In [60]:
print(f"File 2 successful: {file2['width_bin_value'].notna().sum()} / {len(file2)}")
print(f"File 2 asset count: {file2['asset_id'].nunique()}")

File 2 successful: 219 / 220
File 2 asset count: 220


In [62]:
import pandas as pd

# fix file 1 by manually assigning correct headers
file1_fixed = pd.read_csv("../results/vlm_bridge_width_gemma.csv", header=None)
file1_fixed.columns = ['asset_id', 'timestamp', 'model', 'response', 'latency_s', 
                        'width_bin_value', 'width_bin_confidence', 'parse_error', 
                        'raw_response', 'error', 'traceback']

print(f"File 1 assets: {file1_fixed['asset_id'].nunique()}")
print(f"File 1 successful: {file1_fixed['width_bin_value'].notna().sum()}")

# check which assets are in file 1 but not file 2
file2 = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv")
missing = set(file1_fixed['asset_id']) - set(file2['asset_id'])
print(f"Assets in file 1 not in file 2: {len(missing)}")

File 1 assets: 54
File 1 successful: 54
Assets in file 1 not in file 2: 54


In [64]:
# get only the missing assets from file 1
file1_missing = file1_fixed[file1_fixed['asset_id'].isin(missing)]

# combine with file 2
combined = pd.concat([file2, file1_missing])
combined = combined.sort_values('width_bin_value', na_position='last')
combined = combined.drop_duplicates('asset_id', keep='first')

combined.to_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", index=False)
print(f"Final: {len(combined)} assets, {combined['width_bin_value'].notna().sum()} successful")

Final: 274 assets, 273 successful


In [68]:
import pandas as pd

# read with python engine which is more lenient
df = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", 
                 engine="python", 
                 on_bad_lines="skip")

print(f"Rows read: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"Successful: {df['width_bin_value'].notna().sum() if 'width_bin_value' in df.columns else 'col missing'}")

# save clean version
df.to_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", index=False)

Rows read: 274
Columns: ['asset_id', 'timestamp', 'model', 'response', 'latency_s', 'width_bin_value', 'width_bin_confidence', 'parse_error', 'raw_response', 'error', 'traceback']
Successful: 273


In [69]:
import pandas as pd

df = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", 
                 engine="python", 
                 on_bad_lines="skip")

# resave with clean formatting
df.to_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", index=False)
print(f"Saved clean file: {len(df)} rows, {df['width_bin_value'].notna().sum()} successful")

Saved clean file: 274 rows, 273 successful


In [70]:
import pandas as pd

# just read first few lines raw
with open("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", "r") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:100]}")
        if i > 5:
            break

Line 0: asset_id,timestamp,model,response,latency_s,width_bin_value,width_bin_confidence,parse_error,raw_res
Line 1: 48695,2026-05-27T21:13:57.941626,gemma-4-26b-a4b-it,"```json

Line 2: {

Line 3:     ""width_bin"": {""value"": ""narrow (<0.8m)"", ""confidence"": 0.6}

Line 4: }

Line 5: ```",27.747,narrow (<0.8m),0.6,,,,

Line 6: 100348,2026-05-27T21:57:29.005207,gemma-4-26b-a4b-it,"```json



In [71]:
import pandas as pd

df = pd.read_csv(
    "../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv",
    engine="python",
    on_bad_lines="skip"
)

# strip newlines from response column
if 'response' in df.columns:
    df['response'] = df['response'].astype(str).str.replace('\n', ' ', regex=False)

df.to_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", index=False)
print(f"Saved: {len(df)} rows")

Saved: 274 rows


In [74]:
import pandas as pd

file2 = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv")
bridge_width = pd.read_csv("../data/processed/train/train_only_bridge_width_assets.csv")

all_assets = set(bridge_width['asset_id'].unique())
covered = set(file2['asset_id'].unique())
missing = all_assets - covered

print(f"Covered: {len(covered)}")
print(f"Missing: {len(missing)}")

missing_df = bridge_width[bridge_width['asset_id'].isin(missing)]
missing_df.to_csv("../data/processed/train/train_only_bridge_width_missing_assets.csv", index=False)

Covered: 220
Missing: 54


In [75]:
file2 = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv")
file3 = pd.read_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma_missing.csv")

combined = pd.concat([file2, file3]).drop_duplicates('asset_id')
combined.to_csv("../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv", index=False)
print(f"Final: {combined['asset_id'].nunique()} assets")

Final: 274 assets


In [76]:
import pandas as pd

files = [
    "../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv",
    "../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv",
    "../results/vlm_bridge_attributes/vlm_bridge_length_gemma.csv",
    "../results/vlm_bridge_attributes/vlm_bridge_abutment_material_gemma.csv",
    "../results/vlm_bridge_attributes/vlm_bridge_pedestrian_railing_gemma.csv",
    "../results/vlm_bridge_attributes/vlm_bridge_bridge_type_gemma.csv",
]

for f in files:
    try:
        df = pd.read_csv(f)
        print(f"✅ {f}: {len(df)} rows")
    except Exception as e:
        print(f"❌ {f}: {e}")

❌ ../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv: Error tokenizing data. C error: Expected 5 fields in line 29, saw 11

✅ ../results/vlm_bridge_attributes/vlm_bridge_width_gemma.csv: 274 rows
❌ ../results/vlm_bridge_attributes/vlm_bridge_length_gemma.csv: [Errno 2] No such file or directory: '../results/vlm_bridge_attributes/vlm_bridge_length_gemma.csv'
❌ ../results/vlm_bridge_attributes/vlm_bridge_abutment_material_gemma.csv: [Errno 2] No such file or directory: '../results/vlm_bridge_attributes/vlm_bridge_abutment_material_gemma.csv'
❌ ../results/vlm_bridge_attributes/vlm_bridge_pedestrian_railing_gemma.csv: [Errno 2] No such file or directory: '../results/vlm_bridge_attributes/vlm_bridge_pedestrian_railing_gemma.csv'
❌ ../results/vlm_bridge_attributes/vlm_bridge_bridge_type_gemma.csv: [Errno 2] No such file or directory: '../results/vlm_bridge_attributes/vlm_bridge_bridge_type_gemma.csv'


In [77]:
import pandas as pd

df = pd.read_csv(
    "../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv",
    engine="python",
    on_bad_lines="skip"
)

# drop rows where fall_height_bin_value is missing (failed first run rows)
df = df[df["fall_height_bin_value"].notna()]

print(f"Rows after cleaning: {len(df)}")
print(f"Successful: {df['fall_height_bin_value'].notna().sum()}")

df.to_csv("../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv", index=False)
print("Saved clean file")

KeyError: 'fall_height_bin_value'

In [78]:
import pandas as pd

df = pd.read_csv(
    "../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.columns.tolist())
print(df.head(3))

['asset_id', 'timestamp', 'model', 'error', 'traceback']
   asset_id                   timestamp               model  error  traceback
0     52457  2026-05-27T20:59:51.565554  gemma-4-26b-a4b-it    NaN        NaN
1     59060  2026-05-27T20:59:52.311724  gemma-4-26b-a4b-it    NaN        NaN
2     63068  2026-05-27T20:59:52.543129  gemma-4-26b-a4b-it    NaN        NaN


In [80]:
import pandas as pd

# read with correct column names since file has 2 different schemas concatenated
df = pd.read_csv(
    "../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.columns.tolist())
print(f"Shape: {df.shape}")

# the first 27 rows have 5 cols, rest have 11
# find which column has the fall height values
for col in df.columns:
    print(f"{col}: {df[col].dropna().head(3).tolist()}")

['asset_id', 'timestamp', 'model', 'error', 'traceback']
Shape: (27, 5)
asset_id: [52457, 59060, 63068]
timestamp: ['2026-05-27T20:59:51.565554', '2026-05-27T20:59:52.311724', '2026-05-27T20:59:52.543129']
model: ['gemma-4-26b-a4b-it', 'gemma-4-26b-a4b-it', 'gemma-4-26b-a4b-it']
error: []
traceback: []


In [81]:
import pandas as pd

# read all lines raw
with open("../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv", "r") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"Line 0: {lines[0]}")
print(f"Line 27: {lines[27]}")
print(f"Line 28: {lines[28]}")

Total lines: 156
Line 0: asset_id,timestamp,model,error,traceback

Line 27: 129739,2026-05-27T21:00:05.985684,gemma-4-26b-a4b-it,,

Line 28: 52457,2026-05-27T21:01:39.248297,gemma-4-26b-a4b-it,"```json



In [82]:
import pandas as pd

# read only second run (lines 28 onwards) with correct header
df = pd.read_csv(
    "../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv",
    engine="python",
    on_bad_lines="skip",
    skiprows=28,
    names=["asset_id", "timestamp", "model", "response", "latency_s", 
           "fall_height_bin_value", "fall_height_bin_confidence", 
           "parse_error", "raw_response", "error", "traceback"]
)

print(f"Rows: {len(df)}")
print(f"Successful: {df['fall_height_bin_value'].notna().sum()}")

df.to_csv("../results/vlm_bridge_attributes/vlm_bridge_fall_height_gemma.csv", index=False)
print("Saved clean file")

Rows: 27
Successful: 27
Saved clean file


## VLM testing continued

In [2]:
import pandas as pd

part1 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete.csv", engine="python", on_bad_lines="skip")
part2 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part2.csv", engine="python", on_bad_lines="skip")
part3 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part3.csv", engine="python", on_bad_lines="skip")
part4 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part3_part2.csv", engine="python", on_bad_lines="skip")
part5 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part4.csv", engine="python", on_bad_lines="skip")

combined = pd.concat([part1, part2, part3, part4, part5 ]).drop_duplicates("asset_id")
combined.to_csv("../results/vlm_bridge_attributes/vlm_trail_bridge_gemma-4-26b_complete.csv", index=False)
print(f"Combined: {combined['asset_id'].nunique()} assets")

## command to run after merge
#python scripts/evaluate_predictions.py --predictions results/vlm_trail_bridge_gemma-4-26b_complete.csv --ground_truth_dir data/processed/train --attributes fall_height_bin length_bin width_bin attr_has_pedestrian_railing attr_abutment_material attr_bridge_type attr_decking_material --model gemma-4-26b-a4b-it --asset_type "Trail Bridge" --prompt_version trail_bridge_v1


Combined: 304 assets


In [4]:
import pandas as pd

part1 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete.csv", engine="python", on_bad_lines="skip")
part2 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part2.csv", engine="python", on_bad_lines="skip")
part3 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part3.csv", engine="python", on_bad_lines="skip")
part3_part2 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part3_part2.csv", engine="python", on_bad_lines="skip")
part4 = pd.read_csv("../results/vlm_trail_bridge_gemma-4-26b_complete_part4.csv", engine="python", on_bad_lines="skip")

print(f"Part 3 part 2: {part3_part2['asset_id'].nunique()}")

covered = set(part1["asset_id"]) | set(part2["asset_id"]) | set(part3["asset_id"]) | set(part3_part2["asset_id"]) | set(part4["asset_id"])

bridge_input = pd.read_csv("../data/processed/train/train_only_bridge.csv")
missing = bridge_input[~bridge_input["asset_id"].isin(covered)]
print(f"Total unique across all parts: {len(covered)}")
print(f"Missing assets: {missing['asset_id'].nunique()}")
#missing.to_csv("../data/processed/train/train_only_bridge_missing.csv", index=False)

KeyError: 'asset_id'

In [ ]:
## has edge guard dataset creation

import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

# attribute-specific: boardwalk < 1.2m with edge guard labels only
boardwalk_edge_guard = train_only[
    (train_only["profile_name"] == "Boardwalk < 1.2m High") &
    (train_only["attr_has_edge_guard"].notna())
]
print(f"Boardwalk < 1.2m with edge guard label: {boardwalk_edge_guard['asset_id'].nunique()}")
boardwalk_edge_guard.to_csv("../data/processed/train/train_only_boardwalk_edge_guard_assets.csv", index=False)

# asset-specific: all boardwalk < 1.2m assets
boardwalk_low = train_only[
    train_only["profile_name"] == "Boardwalk < 1.2m High"
]
print(f"Boardwalk < 1.2m total assets: {boardwalk_low['asset_id'].nunique()}")
#boardwalk_low.to_csv("../data/processed/train/train_only_boardwalk_low_assets.csv", index=False)
